# 🎓 Career Learning Path Generator — Multi-Agent System

Welcome! This notebook builds a **multi-agent learning path advisor** powered by Google ADK. Given a user's:

- 🎯 **Career Path** (10 options)
- 📊 **Current Level** (Beginner / Intermediate / Advanced)
- ⏱️ **Time Availability** (hours per week or total months)

...it generates a structured, personalised learning roadmap.

### Architecture Overview

```
                    +---------------------+
                    |    User Input 🗣️     |
                    | career / level / time|
                    +----------+----------+
                               |
                               v
                    +---------------------+
                    |   Router Agent 🧠    |
                    | (Selects workflow)  |
                    +----------+----------+
                               |
          +--------------------+----------------------+
          |                    |                      |
          v                    v                      v
  +---------------+  +-------------------+  +--------------------+
  | skills_agent  |  | resources_agent   |  | timeline_agent     |
  | (ParallelAgent|  | (ParallelAgent)   |  | (SequentialAgent)  |
  | fetch skills) |  | fetch resources   |  | phased schedule    |
  +---------------+  +-------------------+  +--------------------+
          |                    |                      |
          +--------------------+----------------------+
                               |
                               v
                    +---------------------+
                    |  synthesis_agent 📋  |
                    | (Final roadmap)     |
                    +---------------------+
```

---

## Part 0: Setup & Authentication 🔑

In [ ]:
!pip install google-adk google-generativeai -q

import os
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
from google.adk.agents import Agent, SequentialAgent, ParallelAgent
from google.adk.tools import google_search
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

print("✅ All libraries are ready!")

In [ ]:
# --- Securely Configure Your API Key ---
api_key = getpass('Enter your Google API Key: ')
genai.configure(api_key=api_key)
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured!")

## Part 1: Helper Utilities & Session Setup

In [ ]:
# --- Supported career paths ---
CAREER_PATHS = {
    "1": "Software Developer / Engineer",
    "2": "AI / Machine Learning Engineer",
    "3": "Data Scientist",
    "4": "Cybersecurity Analyst / Engineer",
    "5": "Cloud Architect",
    "6": "DevOps Engineer",
    "7": "Data Engineer",
    "8": "Web Developer",
    "9": "Computer Systems Analyst",
    "10": "Software Architect",
}

LEVELS = ["beginner", "intermediate", "advanced"]

# --- Session Service (shared across all agents) ---
session_service = InMemorySessionService()
my_user_id = "learner_001"

# --- A reusable runner helper ---
async def run_agent_query(
    agent: Agent,
    query: str,
    session: Session,
    user_id: str,
    silent: bool = False
):
    """Runs a query through a given agent and returns the final response text."""
    if not silent:
        print(f"\n🚀 Running '{agent.name}' in session '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"⚠️ Error: {e}"

    return final_response


def validate_inputs(career_choice: str, level: str, time_input: str) -> dict:
    """
    Validates and normalises user inputs.
    Returns a dict with 'career', 'level', 'time_description' or raises ValueError.
    """
    if career_choice not in CAREER_PATHS:
        raise ValueError(
            f"Invalid career choice '{career_choice}'. "
            f"Please choose a number between 1 and 10."
        )

    level = level.strip().lower()
    if level not in LEVELS:
        raise ValueError(
            f"Invalid level '{level}'. "
            f"Must be one of: {', '.join(LEVELS)}."
        )

    time_input = time_input.strip().lower()
    if "month" in time_input:
        time_description = time_input  # e.g. "6 months"
    elif "hour" in time_input or "hr" in time_input:
        time_description = time_input  # e.g. "10 hours per week"
    elif time_input.replace(".", "").isdigit():
        # Bare number — treat as hours/week
        time_description = f"{time_input} hours per week"
    else:
        raise ValueError(
            "Could not understand time input. "
            "Please provide something like '6 months' or '10 hours per week'."
        )

    return {
        "career": CAREER_PATHS[career_choice],
        "level": level,
        "time_description": time_description,
    }


def build_user_query(params: dict) -> str:
    """Turns validated params into a natural-language query for the agents."""
    return (
        f"Career goal: {params['career']}. "
        f"Current level: {params['level']}. "
        f"Available time: {params['time_description']}."
    )


print("✅ Utilities ready!")

## Part 2: Specialist Agent Definitions

We define four specialist agents that run **in parallel** to research independent aspects of the learning plan:

| Agent | Responsibility | State key |
|---|---|---|
| `skills_agent` | Core skills & technologies to learn | `skills_result` |
| `resources_agent` | Best courses, books, & platforms | `resources_result` |
| `projects_agent` | Hands-on project ideas | `projects_result` |
| `market_agent` | Job market context & salary info | `market_result` |

In [ ]:
# ── Specialist Agent 1: Skills ────────────────────────────────────────────────
skills_agent = Agent(
    name="skills_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Identifies the essential skills and technologies for a given tech career path.",
    instruction="""
You are a senior tech career coach specialising in skill mapping.

Given a career goal, current level, and available time, produce a concise, structured list of:
1. **Foundation skills** the learner must master first (prioritised for their current level).
2. **Intermediate skills** to progress toward after foundations.
3. **Advanced / specialisation skills** to aim for eventually.

Use Google Search to verify current industry expectations and trending tools for {career}.
Tailor depth to the stated level: {level}.
Keep the total output under 400 words — be specific, not generic.

Format as a Markdown section titled "## 🛠️ Skills Roadmap".
""",
    output_key="skills_result"
)

# ── Specialist Agent 2: Resources ─────────────────────────────────────────────
resources_agent = Agent(
    name="resources_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Finds the best learning resources for a specific tech career path and level.",
    instruction="""
You are an expert learning curator for tech careers.

Given a career goal, current level, and time budget, recommend a curated list of the best
learning resources. Include a mix of:
- Online courses (Coursera, Udemy, edX, YouTube, etc.)
- Books or documentation
- Interactive platforms (LeetCode, Kaggle, HackTheBox, etc.) where relevant
- Official certifications worth pursuing

Career: {career}. Level: {level}. Time available: {time_description}.

Use Google Search to find current, highly-rated resources (avoid outdated ones).
For each resource, mention the approximate time commitment and cost (free / paid).
Keep output under 400 words.

Format as a Markdown section titled "## 📚 Learning Resources".
""",
    output_key="resources_result"
)

# ── Specialist Agent 3: Hands-on Projects ─────────────────────────────────────
projects_agent = Agent(
    name="projects_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Suggests portfolio-building projects tailored to a career path and level.",
    instruction="""
You are a senior software mentor who designs portfolio-building projects.

Given a career goal and current level, suggest 4–6 hands-on projects that will:
- Reinforce theoretical knowledge with practice
- Be appropriate in complexity for the learner's level
- Be impressive to hiring managers in the {career} field

For each project, provide:
- A short title and one-sentence description
- Key technologies/skills it practises
- Estimated time to complete

Level: {level}. Use Google Search to confirm relevance to current hiring trends.
Keep output under 400 words.

Format as a Markdown section titled "## 🏗️ Project Ideas".
""",
    output_key="projects_result"
)

# ── Specialist Agent 4: Job Market Context ────────────────────────────────────
market_agent = Agent(
    name="market_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Provides job market context, salary ranges, and hiring tips for a tech career.",
    instruction="""
You are a tech recruitment specialist with deep market knowledge.

Given a career path, provide a concise overview of:
1. **Current demand** — is it growing, stable, or declining?
2. **Typical salary range** — entry, mid, senior (use current data).
3. **Top industries / companies** hiring for this role.
4. **Must-have credentials** — certifications, degrees, portfolio signals.

Career: {career}. Use Google Search for the most current market data.
Keep output under 350 words.

Format as a Markdown section titled "## 💼 Job Market Snapshot".
""",
    output_key="market_result"
)

print("✅ Four specialist agents defined!")

## Part 3: Timeline Agent (Sequential)

This agent builds a **phased week-by-week / month-by-month schedule** based on the skills already researched. It uses `{skills_result}` from the shared state.

In [ ]:
timeline_agent = Agent(
    name="timeline_agent",
    model="gemini-2.5-flash",
    tools=[google_search],
    description="Creates a phased learning timeline based on identified skills and time constraints.",
    instruction="""
You are a learning design specialist who creates realistic, phased study schedules.

You have been given:
- Career goal: {career}
- Current level: {level}
- Available time: {time_description}
- Skills roadmap (already researched): {skills_result}

Using this context, build a **phased timeline** that:
1. Breaks the total available time into logical phases (e.g. Phase 1 / 2 / 3).
2. Assigns specific skills and topics to each phase.
3. Sets a clear milestone / checkpoint goal at the end of each phase.
4. Accounts for realistic study hours per week vs. content volume.

Be specific about durations (e.g. "Weeks 1–4: Python fundamentals — 10 hrs/week").
Keep total output under 500 words.

Format as a Markdown section titled "## 📅 Phased Timeline".
""",
    output_key="timeline_result"
)

print("✅ Timeline agent defined!")

## Part 4: Synthesis Agent

The final agent reads **all four parallel results + the timeline** from shared state and assembles a polished, comprehensive learning roadmap.

In [ ]:
synthesis_agent = Agent(
    name="synthesis_agent",
    model="gemini-2.5-flash",
    description="Synthesises all research into a comprehensive, well-formatted learning roadmap.",
    instruction="""
You are a senior learning consultant who compiles personalised career roadmaps.

Combine the following research sections into one cohesive, motivating learning roadmap document.

--- INPUT SECTIONS ---
{skills_result}

{timeline_result}

{resources_result}

{projects_result}

{market_result}
--- END INPUT SECTIONS ---

Instructions for the final document:
1. Open with a personalised **executive summary** (3–4 sentences) that addresses the learner directly.
   Mention their career goal ({career}), current level ({level}), and time budget ({time_description}).
2. Include all five sections in a logical reading order:
   Job Market → Skills Roadmap → Phased Timeline → Learning Resources → Project Ideas
3. Add a short **"Quick Wins for Week 1"** section at the end with 3 concrete, actionable steps 
   the learner can take immediately.
4. Keep the tone encouraging and professional.
5. Format the whole document cleanly in Markdown.
"""
)

print("✅ Synthesis agent defined!")

## Part 5: Assemble the Multi-Agent Workflow

```
learning_roadmap_agent  (SequentialAgent)
│
├── parallel_research_agent  (ParallelAgent — runs 4 agents simultaneously)
│   ├── skills_agent
│   ├── resources_agent
│   ├── projects_agent
│   └── market_agent
│
├── timeline_agent  (reads skills_result from state)
│
└── synthesis_agent  (reads all results from state → final roadmap)
```

In [ ]:
# ── Step 1: Parallel research phase ───────────────────────────────────────────
parallel_research_agent = ParallelAgent(
    name="parallel_research_agent",
    sub_agents=[skills_agent, resources_agent, projects_agent, market_agent],
    description="Runs all four research agents concurrently to gather comprehensive career info."
)

# ── Step 2: Sequential orchestration (research → timeline → synthesis) ─────────
learning_roadmap_agent = SequentialAgent(
    name="learning_roadmap_agent",
    sub_agents=[
        parallel_research_agent,  # Phase A: parallel research
        timeline_agent,           # Phase B: build timeline using skills
        synthesis_agent,          # Phase C: compile final roadmap
    ],
    description="End-to-end workflow that produces a personalised career learning roadmap."
)

print("✅ Full multi-agent workflow assembled!")
print()
print("Workflow:")
print("  learning_roadmap_agent (SequentialAgent)")
print("  ├── parallel_research_agent (ParallelAgent)")
print("  │   ├── skills_agent")
print("  │   ├── resources_agent")
print("  │   ├── projects_agent")
print("  │   └── market_agent")
print("  ├── timeline_agent")
print("  └── synthesis_agent")

## Part 6: Interactive Input & Execution

Run the cell below to enter your details and receive your personalised roadmap.

In [ ]:
# ── Display career path menu ───────────────────────────────────────────────────
print("=" * 55)
print("       🎓 CAREER LEARNING PATH GENERATOR")
print("=" * 55)
print()
print("Available Career Paths:")
for k, v in CAREER_PATHS.items():
    print(f"  {k:>2}. {v}")
print()
print("Levels: beginner | intermediate | advanced")
print("Time  : e.g.  '6 months'  |  '10 hours per week'  |  '200 hours'")
print("=" * 55)
print()

# ── Collect user inputs ────────────────────────────────────────────────────────
career_choice = input("Enter career number (1-10): ").strip()
level         = input("Enter your level (beginner/intermediate/advanced): ").strip()
time_input    = input("Enter available time (e.g. '3 months' or '8 hours per week'): ").strip()

# ── Validate ───────────────────────────────────────────────────────────────────
try:
    params = validate_inputs(career_choice, level, time_input)
except ValueError as err:
    print(f"\n❌ Input error: {err}")
    raise

user_query = build_user_query(params)

print()
print("✅ Inputs accepted!")
print(f"   Career : {params['career']}")
print(f"   Level  : {params['level']}")
print(f"   Time   : {params['time_description']}")
print()
print(f"📝 Query sent to agents: '{user_query}'")

In [ ]:
# ── Run the full multi-agent workflow ──────────────────────────────────────────

async def generate_learning_roadmap(query: str, params: dict):
    print(f"\n{'='*60}")
    print(f"🚀 Generating roadmap for: {params['career']}")
    print(f"   Level: {params['level']} | Time: {params['time_description']}")
    print(f"{'='*60}\n")

    print("⚡ Step 1/3 — Running parallel research agents (skills, resources, projects, market)...")
    print("⏳ Step 2/3 — Timeline agent will run after parallel research completes...")
    print("📋 Step 3/3 — Synthesis agent will compile your final roadmap...")
    print()

    session = await session_service.create_session(
        app_name=learning_roadmap_agent.name,
        user_id=my_user_id
    )

    roadmap = await run_agent_query(
        learning_roadmap_agent,
        query,
        session,
        my_user_id,
        silent=True
    )

    print("\n" + "=" * 60)
    print("✅ YOUR PERSONALISED LEARNING ROADMAP")
    print("=" * 60 + "\n")
    display(Markdown(roadmap))
    return roadmap


roadmap_output = await generate_learning_roadmap(user_query, params)

## Part 7: Batch Mode — Generate Multiple Roadmaps at Once

Need roadmaps for several profiles? Use the helper below to run them in sequence.

In [ ]:
async def batch_roadmaps(profiles: list[dict]):
    """
    Generate roadmaps for a list of profiles.
    Each profile is a dict: {career_choice, level, time_input}
    """
    results = []
    for i, profile in enumerate(profiles, 1):
        print(f"\n{'#'*60}")
        print(f"  Profile {i}/{len(profiles)}")
        print(f"{'#'*60}")
        try:
            p = validate_inputs(
                profile["career_choice"],
                profile["level"],
                profile["time_input"]
            )
            q = build_user_query(p)
            roadmap = await generate_learning_roadmap(q, p)
            results.append({"params": p, "roadmap": roadmap})
        except ValueError as err:
            print(f"❌ Skipping profile {i}: {err}")
    return results


# ── Example batch usage (uncomment and customise) ──────────────────────────────
# example_profiles = [
#     {"career_choice": "2", "level": "beginner",      "time_input": "6 months"},
#     {"career_choice": "4", "level": "intermediate",   "time_input": "10 hours per week"},
#     {"career_choice": "5", "level": "advanced",       "time_input": "3 months"},
# ]
# batch_results = await batch_roadmaps(example_profiles)

print("✅ Batch mode ready — uncomment the example above to run it.")

---
## 🎉 Congratulations!

You have built a fully functional **multi-agent career learning path generator** using Google ADK.

### What was demonstrated:

- **`ParallelAgent`** — Four specialist research agents (skills, resources, projects, market) run concurrently, dramatically speeding up information gathering.
- **`SequentialAgent`** — The timeline agent and synthesis agent run in strict order *after* parallel research, consuming shared state populated by earlier agents.
- **Shared State via `output_key`** — Each specialist agent saves its result to a named key (`skills_result`, `resources_result`, etc.), which downstream agents access via `{placeholder}` syntax in their instructions.
- **Input Validation** — A helper layer validates and normalises user inputs before any agent is invoked.
- **Batch Mode** — Multiple learner profiles can be processed sequentially.

```
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )
   > 🎓 <    > 📚 <    > 💻 <    > 🛠️  <     > 🚀 <
```